# Data Anomaly Detection Report

Ingests any time series dataset with an ID column and a timestamp column.  
Detects: structural integrity issues, value outliers, temporal pattern breaks, and cross-entity anomalies.  
Outputs: prioritised findings table, visualisations, and an optional markdown report file.

**Only Cell 2 (CONFIG) needs to be edited for each dataset.**

## Section 1 — Setup

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import pathlib
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('pandas', pd.__version__, '| polars', pl.__version__, '| numpy', np.__version__)

In [ ]:
# ============================================================
# USER CONFIGURATION  ←  Edit this cell for your dataset
# ============================================================
CONFIG = {
    # --- DATA SOURCE ---
    # Accepts: .csv, .parquet, .json, .xlsx — any pandas-readable file
    # Or set file_path=None and assign df_override=your_dataframe below
    'file_path': 'sample_data.csv',
    'df_override': None,           # set to a pandas DataFrame to skip file loading

    # Column names
    'id_col':    'entity_id',      # column identifying each entity / series
    'time_col':  'timestamp',      # datetime or ISO-parseable string column
    'value_cols': ['value'],       # list of numeric columns to analyse

    # --- TEMPORAL EXPECTATIONS ---
    'expected_freq': None,         # pandas offset alias e.g. '1h','15min' — None = auto-infer
    'max_gap_multiplier': 3,       # gaps > N * median_interval are flagged as structural gaps

    # --- STATISTICAL THRESHOLDS ---
    'zscore_threshold': 3.5,              # |Z| > threshold  (standard Z-score per ID)
    'modified_zscore_threshold': 3.5,     # |modified Z| > threshold  (MAD-based, robust)
    'iqr_multiplier': 1.5,                # Tukey fence multiplier  (k=1.5 outer, k=3.0 far)
    'rolling_window': 24,                 # rolling window in rows
    'rolling_zscore_threshold': 3.0,      # local spike threshold
    'level_shift_threshold': 2.0,         # sigma threshold for rolling mean shift
    'trend_break_threshold': 3.0,         # OLS residual sigma threshold
    'volatility_ratio_threshold': 4.0,    # vol_after / vol_before threshold

    # --- CROSS-ENTITY ---
    'cross_entity_zscore': 3.0,           # entity-level aggregate Z-score threshold

    # --- OUTPUT ---
    'report_top_n': 20,
    'plot_anomalies': True,
    'output_report_path': 'anomaly_report.md',  # set None to skip file output
}

In [ ]:
# Anomaly registry — all detectors write here via register()
anomaly_registry = {
    'structural':   [],
    'value':        [],
    'temporal':     [],
    'cross_entity': [],
}

def register(category: str, record: dict):
    """Append a standardised anomaly record to the registry."""
    required = {'id', 'time', 'col', 'value', 'method', 'score', 'severity', 'label'}
    missing = required - record.keys()
    if missing:
        raise ValueError(f'register() missing fields: {missing}')
    record['category'] = category
    anomaly_registry[category].append(record)

print('Registry initialised.')

## Section 2 — Data Ingestion

In [ ]:
# Load data
if CONFIG['df_override'] is not None:
    raw_pd = CONFIG['df_override'].copy()
    print('Using df_override.')
else:
    fp = pathlib.Path(CONFIG['file_path'])
    ext = fp.suffix.lower()
    if ext == '.csv':     raw_pd = pd.read_csv(fp)
    elif ext == '.parquet': raw_pd = pd.read_parquet(fp)
    elif ext == '.json':  raw_pd = pd.read_json(fp)
    elif ext in ('.xlsx','.xls'): raw_pd = pd.read_excel(fp)
    else: raise ValueError(f'Unsupported file format: {ext}')
    print(f'Loaded from {fp}')

# Validate required columns
for col_key, col_name in [('id_col', CONFIG['id_col']),
                           ('time_col', CONFIG['time_col'])]:
    assert col_name in raw_pd.columns, f"Required column '{col_name}' not found. Columns: {list(raw_pd.columns)}"
for vc in CONFIG['value_cols']:
    assert vc in raw_pd.columns, f"Value column '{vc}' not found. Columns: {list(raw_pd.columns)}"

# Parse timestamps
if not pd.api.types.is_datetime64_any_dtype(raw_pd[CONFIG['time_col']]):
    raw_pd[CONFIG['time_col']] = pd.to_datetime(raw_pd[CONFIG['time_col']], infer_datetime_format=True)

# Cast value columns to float
for vc in CONFIG['value_cols']:
    raw_pd[vc] = pd.to_numeric(raw_pd[vc], errors='coerce')

n_ids = raw_pd[CONFIG['id_col']].nunique()
print(f'{len(raw_pd):,} rows | {n_ids} unique IDs | {raw_pd[CONFIG["time_col"]].min()} → {raw_pd[CONFIG["time_col"]].max()}')
raw_pd.dtypes

In [ ]:
# Convert to Polars for all downstream processing
df = pl.from_pandas(raw_pd)
df = df.sort([CONFIG['id_col'], CONFIG['time_col']])

for vc in CONFIG['value_cols']:
    df = df.with_columns(pl.col(vc).cast(pl.Float64))

print('Polars schema:')
print(df.schema)
df.head(5)

In [ ]:
# Infer or set median interval in seconds
def infer_median_interval(df: pl.DataFrame, id_col: str, time_col: str) -> float:
    intervals = (
        df.sort([id_col, time_col])
          .with_columns(
              pl.col(time_col).diff().over(id_col).alias('_diff')
          )
          .filter(pl.col('_diff').is_not_null())
          .select(pl.col('_diff').dt.total_seconds())
          .to_series().to_list()
    )
    return float(np.median([x for x in intervals if x > 0]))

if CONFIG['expected_freq'] is None:
    median_interval_s = infer_median_interval(df, CONFIG['id_col'], CONFIG['time_col'])
    print(f'Auto-inferred median interval: {median_interval_s:.1f}s ({median_interval_s/3600:.2f}h)')
else:
    median_interval_s = pd.tseries.frequencies.to_offset(CONFIG['expected_freq']).nanos / 1e9
    print(f'Configured interval: {median_interval_s:.1f}s')

## Section 3 — Structural Integrity Checks

In [ ]:
# Duplicate timestamp detection
dup_check = (
    df.group_by([CONFIG['id_col'], CONFIG['time_col']])
      .agg(pl.len().alias('row_count'))
      .filter(pl.col('row_count') > 1)
      .sort([CONFIG['id_col'], CONFIG['time_col']])
)

for row in dup_check.to_dicts():
    register('structural', {
        'id':    row[CONFIG['id_col']],
        'time':  str(row[CONFIG['time_col']]),
        'col':   CONFIG['time_col'],
        'value': row['row_count'],
        'method': 'duplicate_timestamp',
        'score':  float(row['row_count']),
        'severity': 9,
        'label': f"{row['row_count']} rows share this timestamp — likely a pipeline bug",
    })

print(f'Duplicate timestamp groups: {len(dup_check)}')
if len(dup_check): display(dup_check)

In [ ]:
# Missing value audit
null_counts = df.select([
    pl.col(c).null_count().alias(c) for c in CONFIG['value_cols']
])
null_pct = df.select([
    (pl.col(c).null_count() / pl.len() * 100).round(2).alias(c)
    for c in CONFIG['value_cols']
])
print('Null counts:'); display(null_counts)
print('Null %:');     display(null_pct)

In [ ]:
# Temporal gap detection — flag gaps > max_gap_multiplier * median_interval
gap_threshold_s = CONFIG['max_gap_multiplier'] * median_interval_s

gaps = (
    df.sort([CONFIG['id_col'], CONFIG['time_col']])
      .with_columns(
          pl.col(CONFIG['time_col'])
            .diff().over(CONFIG['id_col'])
            .dt.total_seconds()
            .alias('_gap_s')
      )
      .filter(pl.col('_gap_s') > gap_threshold_s)
      .select([CONFIG['id_col'], CONFIG['time_col'], '_gap_s'])
)

for row in gaps.to_dicts():
    gap_multiple = row['_gap_s'] / median_interval_s
    register('structural', {
        'id':    row[CONFIG['id_col']],
        'time':  str(row[CONFIG['time_col']]),
        'col':   CONFIG['time_col'],
        'value': round(row['_gap_s'], 1),
        'method': 'temporal_gap',
        'score':  round(gap_multiple, 2),
        'severity': min(10, int(3 + gap_multiple)),
        'label': f"Gap of {row['_gap_s']:.0f}s ({gap_multiple:.1f}× expected interval)",
    })

print(f'Temporal gaps detected: {len(gaps)}')
if len(gaps): display(gaps)

In [ ]:
# Short series detection — entities with < 10% of max series length
series_lengths = (
    df.group_by(CONFIG['id_col'])
      .agg(pl.len().alias('n_rows'))
      .sort('n_rows')
)
max_len = series_lengths['n_rows'].max()
median_len = series_lengths['n_rows'].median()
short_threshold = 0.10 * max_len

for row in series_lengths.filter(pl.col('n_rows') < short_threshold).to_dicts():
    register('structural', {
        'id':    row[CONFIG['id_col']],
        'time':  '—',
        'col':   'series_length',
        'value': row['n_rows'],
        'method': 'short_series',
        'score':  round(float(median_len) / max(row['n_rows'], 1), 2),
        'severity': 6,
        'label': f"Only {row['n_rows']} rows vs median {median_len:.0f} — possible ingestion truncation",
    })

print(f'Series lengths (max={max_len}, median={median_len:.0f}):')
display(series_lengths)

In [ ]:
# Structural summary visualisation
if CONFIG['plot_anomalies'] and len(gaps) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle('Structural Integrity Overview', fontsize=13, fontweight='bold')

    # Left: gap timeline
    ax = axes[0]
    gaps_pd = gaps.to_pandas()
    gaps_pd[CONFIG['time_col']] = pd.to_datetime(gaps_pd[CONFIG['time_col']])
    entities = gaps_pd[CONFIG['id_col']].unique()
    y_map = {e: i for i, e in enumerate(entities)}
    sc = ax.scatter(
        gaps_pd[CONFIG['time_col']],
        gaps_pd[CONFIG['id_col']].map(y_map),
        c=gaps_pd['_gap_s'], cmap='OrRd', s=80, zorder=3
    )
    ax.set_yticks(range(len(entities)))
    ax.set_yticklabels(entities)
    ax.set_title('Temporal Gaps by Entity'); ax.set_xlabel('Time')
    plt.colorbar(sc, ax=ax, label='Gap (seconds)')

    # Right: series lengths
    ax2 = axes[1]
    sl_pd = series_lengths.to_pandas()
    colours = ['crimson' if v < short_threshold else 'steelblue' for v in sl_pd['n_rows']]
    ax2.barh(sl_pd[CONFIG['id_col']], sl_pd['n_rows'], color=colours)
    ax2.axvline(short_threshold, color='red', linestyle='--', label='10% threshold')
    ax2.set_title('Series Lengths'); ax2.set_xlabel('Row count')
    ax2.legend()

    plt.tight_layout()
    plt.show()
else:
    print('No structural gaps to plot or plotting disabled.')

## Section 4 — Value-Based Anomaly Detection

In [ ]:
# Method 1: Standard Z-score per ID
# Z = (x - mean_id) / std_id
# Fast; assumes approximate normality. Baseline method.

z_hits = {}  # (id, time, col) -> abs_z for consensus scoring

for vc in CONFIG['value_cols']:
    flagged = (
        df.with_columns([
            pl.col(vc).mean().over(CONFIG['id_col']).alias('_mean'),
            pl.col(vc).std().over(CONFIG['id_col']).alias('_std'),
        ])
        .with_columns(
            ((pl.col(vc) - pl.col('_mean')) / pl.col('_std')).alias('_z')
        )
        .filter(pl.col('_z').abs() > CONFIG['zscore_threshold'])
        .select([CONFIG['id_col'], CONFIG['time_col'], vc, '_z'])
    )
    for row in flagged.to_dicts():
        key = (row[CONFIG['id_col']], str(row[CONFIG['time_col']]), vc)
        z_hits[key] = abs(row['_z'])
        register('value', {
            'id':    row[CONFIG['id_col']],
            'time':  str(row[CONFIG['time_col']]),
            'col':   vc,
            'value': round(row[vc], 4) if row[vc] is not None else None,
            'method': 'zscore',
            'score':  round(abs(row['_z']), 3),
            'severity': 5,
            'label': f'Z-score {row["_z"]:.2f} (threshold ±{CONFIG["zscore_threshold"]})',
        })

print(f'Standard Z-score flags: {len(z_hits)}')

In [ ]:
# Method 2: Modified Z-score (MAD-based) per ID
# MAD = median(|x - median(x)|)
# Modified_Z = 0.6745 * (x - median(x)) / MAD
# Robust: breakdown point 50% vs 0% for standard Z-score.

mz_hits = {}

for vc in CONFIG['value_cols']:
    flagged = (
        df.with_columns(
            pl.col(vc).median().over(CONFIG['id_col']).alias('_med')
        )
        .with_columns(
            (pl.col(vc) - pl.col('_med')).abs().alias('_abs_dev')
        )
        .with_columns(
            pl.col('_abs_dev').median().over(CONFIG['id_col']).alias('_mad')
        )
        .with_columns(
            (0.6745 * (pl.col(vc) - pl.col('_med')) / pl.col('_mad')).alias('_mz')
        )
        .filter(pl.col('_mz').abs() > CONFIG['modified_zscore_threshold'])
        .select([CONFIG['id_col'], CONFIG['time_col'], vc, '_mz'])
    )
    for row in flagged.to_dicts():
        key = (row[CONFIG['id_col']], str(row[CONFIG['time_col']]), vc)
        mz_hits[key] = abs(row['_mz'])
        register('value', {
            'id':    row[CONFIG['id_col']],
            'time':  str(row[CONFIG['time_col']]),
            'col':   vc,
            'value': round(row[vc], 4) if row[vc] is not None else None,
            'method': 'modified_zscore',
            'score':  round(abs(row['_mz']), 3),
            'severity': 6,
            'label': f'Modified Z-score {row["_mz"]:.2f} (threshold ±{CONFIG["modified_zscore_threshold"]})',
        })

print(f'Modified Z-score flags: {len(mz_hits)}')

In [ ]:
# Method 3: IQR (Tukey fence) per ID
# lower = Q1 - k*IQR,  upper = Q3 + k*IQR
# Distribution-free; no normality assumption.

iqr_hits = {}

for vc in CONFIG['value_cols']:
    k = CONFIG['iqr_multiplier']
    flagged = (
        df.with_columns([
            pl.col(vc).quantile(0.25).over(CONFIG['id_col']).alias('_q1'),
            pl.col(vc).quantile(0.75).over(CONFIG['id_col']).alias('_q3'),
        ])
        .with_columns(
            (pl.col('_q3') - pl.col('_q1')).alias('_iqr')
        )
        .with_columns([
            (pl.col('_q1') - k * pl.col('_iqr')).alias('_lower'),
            (pl.col('_q3') + k * pl.col('_iqr')).alias('_upper'),
        ])
        .filter(
            (pl.col(vc) < pl.col('_lower')) | (pl.col(vc) > pl.col('_upper'))
        )
        .select([CONFIG['id_col'], CONFIG['time_col'], vc, '_lower', '_upper', '_iqr'])
    )
    for row in flagged.to_dicts():
        key = (row[CONFIG['id_col']], str(row[CONFIG['time_col']]), vc)
        fence_dist = max(
            row[vc] - row['_upper'] if row[vc] > row['_upper'] else 0,
            row['_lower'] - row[vc] if row[vc] < row['_lower'] else 0,
        )
        score = round(fence_dist / max(row['_iqr'], 1e-9), 3)
        iqr_hits[key] = score
        direction = 'above upper' if row[vc] > row['_upper'] else 'below lower'
        register('value', {
            'id':    row[CONFIG['id_col']],
            'time':  str(row[CONFIG['time_col']]),
            'col':   vc,
            'value': round(row[vc], 4) if row[vc] is not None else None,
            'method': 'iqr_fence',
            'score':  score,
            'severity': 5,
            'label': f'IQR fence violation ({direction}): value={row[vc]:.3g}, fence=[{row["_lower"]:.3g}, {row["_upper"]:.3g}]',
        })

print(f'IQR fence flags: {len(iqr_hits)}')

In [ ]:
# Method consensus: observations flagged by 2+ methods get severity bump
all_value_keys = set(z_hits) | set(mz_hits) | set(iqr_hits)

consensus_2 = set()
consensus_3 = set()
for key in all_value_keys:
    count = sum([key in z_hits, key in mz_hits, key in iqr_hits])
    if count == 3: consensus_3.add(key)
    elif count == 2: consensus_2.add(key)

# Apply bumps retroactively
for rec in anomaly_registry['value']:
    key = (rec['id'], rec['time'], rec['col'])
    if key in consensus_3:
        rec['severity'] = min(10, rec['severity'] + 3)
        rec['label'] += ' [ALL 3 METHODS AGREE]'
    elif key in consensus_2:
        rec['severity'] = min(10, rec['severity'] + 2)
        rec['label'] += ' [2 methods agree]'

print(f'Consensus (2-method): {len(consensus_2)} | Consensus (3-method): {len(consensus_3)}')

In [ ]:
# Heuristic anomaly classification: data entry error vs bug vs genuine anomaly
# Applied to the raw polars data, results used to update registry labels

def classify_value_anomaly(vc: str, value: float, series_mean: float,
                            series_std: float, series_median: float) -> tuple[str, int]:
    """Returns (anomaly_type_label, severity_adjustment)."""
    if value is None or np.isnan(value):
        return 'missing_value', 0
    # Exact zero in a series where zero is implausible
    if value == 0.0 and series_mean > 3 * series_std and series_std > 0:
        return 'data_entry_error (suspicious zero)', +2
    # Round number in a series with sub-unit precision
    if value == float(int(value)) and series_std < 0.5 and abs(value - series_median) > series_std:
        return 'data_entry_error (suspicious round number)', +1
    # Orders-of-magnitude outlier
    if series_std > 0 and abs(value - series_mean) > 10 * series_std:
        return 'possible_bug (extreme magnitude)', +2
    return 'genuine_anomaly_candidate', 0

# Compute per-ID per-column stats for classification
id_stats = {}
for vc in CONFIG['value_cols']:
    stats_df = (
        df.group_by(CONFIG['id_col'])
          .agg([
              pl.col(vc).mean().alias('mean'),
              pl.col(vc).std().alias('std'),
              pl.col(vc).median().alias('median'),
          ])
    )
    for row in stats_df.to_dicts():
        id_stats[(row[CONFIG['id_col']], vc)] = (row['mean'], row['std'], row['median'])

# Update value registry records with classification
for rec in anomaly_registry['value']:
    key = (rec['id'], rec['col'])
    if key in id_stats:
        mean, std, median = id_stats[key]
        atype, adj = classify_value_anomaly(rec['col'], rec['value'], mean, std, median)
        rec['anomaly_type'] = atype
        rec['severity'] = min(10, rec['severity'] + adj)
    else:
        rec['anomaly_type'] = 'unknown'

print('Value anomaly classification applied.')

In [ ]:
# Sensor freeze detection: N+ consecutive identical values within a series
FREEZE_THRESHOLD = 3

for vc in CONFIG['value_cols']:
    freeze_df = (
        df.sort([CONFIG['id_col'], CONFIG['time_col']])
          .with_columns([
              pl.col(vc).eq(pl.col(vc).shift(1).over(CONFIG['id_col'])).alias('_same'),
              pl.col(vc).shift(1).over(CONFIG['id_col']).alias('_prev'),
          ])
          .with_columns(
              # cumulative run length of identical consecutive values per ID
              pl.when(pl.col('_same'))
                .then(pl.lit(1)).otherwise(pl.lit(0))
                .cum_sum().over(CONFIG['id_col'])
                .alias('_run')
          )
    )
    # Find rows where run length >= threshold — flag the start of each run
    # Simpler: flag any row that is part of a run of >= FREEZE_THRESHOLD identical values
    freeze_check = (
        df.sort([CONFIG['id_col'], CONFIG['time_col']])
          .with_columns(
              pl.col(vc).shift(FREEZE_THRESHOLD - 1)
                .over(CONFIG['id_col'])
                .alias('_look_back')
          )
          .filter(
              pl.col(vc).eq(pl.col('_look_back')) & pl.col('_look_back').is_not_null()
          )
          .select([CONFIG['id_col'], CONFIG['time_col'], vc])
          .unique([CONFIG['id_col'], vc])
    )
    for row in freeze_check.to_dicts():
        register('value', {
            'id':    row[CONFIG['id_col']],
            'time':  str(row[CONFIG['time_col']]),
            'col':   vc,
            'value': round(row[vc], 4) if row[vc] is not None else None,
            'method': 'sensor_freeze',
            'score':  float(FREEZE_THRESHOLD),
            'severity': 8,
            'anomaly_type': 'possible_bug (sensor freeze)',
            'label': f'Value {row[vc]} repeated {FREEZE_THRESHOLD}+ consecutive times — possible sensor freeze or copy-paste bug',
        })

print(f'Sensor freeze flags: {len(freeze_check)}')

In [ ]:
# Value anomaly visualisation
if CONFIG['plot_anomalies'] and anomaly_registry['value']:
    val_df = pd.DataFrame(anomaly_registry['value'])
    val_df['time'] = pd.to_datetime(val_df['time'], errors='coerce')

    id_list = raw_pd[CONFIG['id_col']].unique()[:20]  # cap at 20
    ncols = min(3, len(id_list))
    nrows = int(np.ceil(len(id_list) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 3.5 * nrows), squeeze=False)
    fig.suptitle('Value Anomalies by Entity', fontsize=13, fontweight='bold')

    type_colours = {
        'data_entry_error (suspicious zero)':          'orange',
        'data_entry_error (suspicious round number)':  'darkorange',
        'possible_bug (extreme magnitude)':            'red',
        'possible_bug (sensor freeze)':                'purple',
        'genuine_anomaly_candidate':                   'gold',
        'unknown':                                     'grey',
    }

    for idx, eid in enumerate(id_list):
        ax = axes[idx // ncols][idx % ncols]
        vc = CONFIG['value_cols'][0]
        entity_pd = raw_pd[raw_pd[CONFIG['id_col']] == eid].sort_values(CONFIG['time_col'])
        ax.plot(entity_pd[CONFIG['time_col']], entity_pd[vc],
                lw=0.8, color='steelblue', alpha=0.7, zorder=1)

        entity_flags = val_df[(val_df['id'] == eid) & (val_df['col'] == vc)]
        for atype, grp in entity_flags.groupby('anomaly_type', dropna=False):
            colour = type_colours.get(str(atype), 'grey')
            ax.scatter(grp['time'], grp['value'], color=colour,
                       s=60, zorder=3, label=str(atype))

        ax.set_title(str(eid), fontsize=10)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=30, labelsize=7)
        if idx == 0: ax.legend(fontsize=7, loc='upper left')

    # Hide unused axes
    for idx in range(len(id_list), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    plt.tight_layout()
    plt.show()

## Section 5 — Temporal Pattern Anomalies

In [ ]:
# Rolling Z-score: local spike detection
# Flags observations that deviate from the local rolling mean by > threshold sigma
# Catches local spikes that are within the global IQR but anomalous in context

W = CONFIG['rolling_window']
rz_hits = set()

for vc in CONFIG['value_cols']:
    rolling_flagged = (
        df.sort([CONFIG['id_col'], CONFIG['time_col']])
          .with_columns([
              pl.col(vc)
                .rolling_mean(window_size=W, center=True, min_periods=max(3, W//4))
                .over(CONFIG['id_col'])
                .alias('_r_mean'),
              pl.col(vc)
                .rolling_std(window_size=W, center=True, min_periods=max(3, W//4))
                .over(CONFIG['id_col'])
                .alias('_r_std'),
          ])
          .with_columns(
              ((pl.col(vc) - pl.col('_r_mean')) / pl.col('_r_std')).alias('_rz')
          )
          .filter(pl.col('_rz').abs() > CONFIG['rolling_zscore_threshold'])
          .select([CONFIG['id_col'], CONFIG['time_col'], vc, '_rz', '_r_mean', '_r_std'])
    )
    for row in rolling_flagged.to_dicts():
        key = (row[CONFIG['id_col']], str(row[CONFIG['time_col']]), vc)
        rz_hits.add(key)
        mag = abs(row['_rz'])
        register('temporal', {
            'id':    row[CONFIG['id_col']],
            'time':  str(row[CONFIG['time_col']]),
            'col':   vc,
            'value': round(row[vc], 4) if row[vc] is not None else None,
            'method': 'rolling_zscore',
            'score':  round(mag, 3),
            'severity': 7 if mag > 5 else 5,
            'label': f'Local spike: rolling Z={row["_rz"]:.2f} (window={W})',
        })

print(f'Rolling Z-score flags: {len(rz_hits)}')

In [ ]:
# Level shift detection: rolling mean difference
# Approximates CUSUM by comparing trailing vs leading rolling means
# shift_score = |mu_after - mu_before| / pooled_std

shift_hits = set()

for vc in CONFIG['value_cols']:
    shift_df = (
        df.sort([CONFIG['id_col'], CONFIG['time_col']])
          .with_columns([
              pl.col(vc)
                .rolling_mean(window_size=W, min_periods=max(3, W//4))
                .over(CONFIG['id_col'])
                .alias('_trail_mean'),
              pl.col(vc)
                .rolling_mean(window_size=W, center=True, min_periods=max(3, W//4))
                .over(CONFIG['id_col'])
                .alias('_center_mean'),
              pl.col(vc)
                .rolling_std(window_size=W, min_periods=max(3, W//4))
                .over(CONFIG['id_col'])
                .alias('_r_std'),
          ])
          .with_columns(
              ((pl.col('_center_mean') - pl.col('_trail_mean')) / pl.col('_r_std')).abs()
               .alias('_shift_z')
          )
          .filter(pl.col('_shift_z') > CONFIG['level_shift_threshold'])
          .select([CONFIG['id_col'], CONFIG['time_col'], vc, '_shift_z',
                   '_trail_mean', '_center_mean'])
    )
    for row in shift_df.to_dicts():
        key = (row[CONFIG['id_col']], str(row[CONFIG['time_col']]), vc)
        shift_hits.add(key)
        register('temporal', {
            'id':    row[CONFIG['id_col']],
            'time':  str(row[CONFIG['time_col']]),
            'col':   vc,
            'value': round(row[vc], 4) if row[vc] is not None else None,
            'method': 'level_shift',
            'score':  round(row['_shift_z'], 3),
            'severity': 6,
            'label': f'Level shift score={row["_shift_z"]:.2f}σ (before≈{row["_trail_mean"]:.3g} → after≈{row["_center_mean"]:.3g})',
        })

print(f'Level shift flags: {len(shift_hits)}')

In [ ]:
# Trend break detection: OLS residual method
# Fit linear trend over rolling window; flag when next point's residual > threshold * in-window std
# Uses pandas + scipy on small numpy arrays — not vectorisable in polars

trend_breaks = []

for vc in CONFIG['value_cols']:
    pdf_all = (
        df.select([CONFIG['id_col'], CONFIG['time_col'], vc])
          .sort([CONFIG['id_col'], CONFIG['time_col']])
          .to_pandas()
    )
    for eid, grp in pdf_all.groupby(CONFIG['id_col']):
        grp = grp.reset_index(drop=True)
        v = grp[vc].values.astype(float)
        x = np.arange(len(v))
        times = grp[CONFIG['time_col']].values
        W2 = CONFIG['rolling_window']
        threshold = CONFIG['trend_break_threshold']
        for i in range(W2, len(v)):
            seg_x = x[i - W2: i]
            seg_v = v[i - W2: i]
            if np.std(seg_v) < 1e-10 or np.any(np.isnan(seg_v)):
                continue
            slope, intercept, *_ = stats.linregress(seg_x, seg_v)
            predicted = slope * x[i] + intercept
            residuals = seg_v - (slope * seg_x + intercept)
            res_std = np.std(residuals)
            if res_std < 1e-10 or np.isnan(v[i]):
                continue
            z = abs(v[i] - predicted) / res_std
            if z > threshold:
                trend_breaks.append({
                    'id':    eid,
                    'time':  str(times[i]),
                    'col':   vc,
                    'value': round(float(v[i]), 4),
                    'method': 'trend_break',
                    'score':  round(z, 3),
                    'severity': 6 if z > 5 else 4,
                    'label': f'Trend break: residual Z={z:.2f} (predicted={predicted:.3g}, actual={v[i]:.3g})',
                })

# Deduplicate — keep max score per (id, time, col)
trend_dedup = {}
for r in trend_breaks:
    key = (r['id'], r['time'], r['col'])
    if key not in trend_dedup or r['score'] > trend_dedup[key]['score']:
        trend_dedup[key] = r

for rec in trend_dedup.values():
    register('temporal', rec)

print(f'Trend break flags (deduplicated): {len(trend_dedup)}')

In [ ]:
# Volatility regime change: rolling std ratio
# Detects when variance suddenly increases or decreases
# vol_ratio = max(std_before, std_after) / min(std_before, std_after)

vol_hits = set()
VR_THRESH = CONFIG['volatility_ratio_threshold']

for vc in CONFIG['value_cols']:
    vol_df = (
        df.sort([CONFIG['id_col'], CONFIG['time_col']])
          .with_columns([
              pl.col(vc)
                .rolling_std(window_size=W, min_periods=max(3, W//4))
                .over(CONFIG['id_col'])
                .alias('_std_before'),
          ])
          .with_columns([
              # "std_after" = leading rolling std approximated by reversing
              pl.col('_std_before').reverse().over(CONFIG['id_col']).alias('_std_after'),
          ])
          .with_columns(
              (pl.max_horizontal('_std_before', '_std_after') /
               (pl.min_horizontal('_std_before', '_std_after') + 1e-9)).alias('_vol_ratio')
          )
          .filter(
              (pl.col('_vol_ratio') > VR_THRESH) &
              pl.col('_std_before').is_not_null() &
              pl.col('_std_after').is_not_null()
          )
          .select([CONFIG['id_col'], CONFIG['time_col'], vc, '_vol_ratio',
                   '_std_before', '_std_after'])
    )
    for row in vol_df.to_dicts():
        key = (row[CONFIG['id_col']], str(row[CONFIG['time_col']]), vc)
        vol_hits.add(key)
        register('temporal', {
            'id':    row[CONFIG['id_col']],
            'time':  str(row[CONFIG['time_col']]),
            'col':   vc,
            'value': round(row[vc], 4) if row[vc] is not None else None,
            'method': 'volatility_change',
            'score':  round(row['_vol_ratio'], 3),
            'severity': 5,
            'label': f'Volatility ratio={row["_vol_ratio"]:.2f} (before σ={row["_std_before"]:.3g}, after σ={row["_std_after"]:.3g})',
        })

print(f'Volatility change flags: {len(vol_hits)}')

In [ ]:
# Temporal anomaly visualisation — rolling band + flagged points
if CONFIG['plot_anomalies']:
    temp_df = pd.DataFrame(anomaly_registry['temporal'])
    if not temp_df.empty:
        temp_df['time'] = pd.to_datetime(temp_df['time'], errors='coerce')

    id_list = raw_pd[CONFIG['id_col']].unique()[:6]
    vc = CONFIG['value_cols'][0]

    fig, axes = plt.subplots(len(id_list), 1,
                             figsize=(14, 3.5 * len(id_list)), squeeze=False)
    fig.suptitle('Temporal Pattern Anomalies', fontsize=13, fontweight='bold')

    method_colours = {
        'rolling_zscore': 'crimson',
        'level_shift':    'darkorange',
        'trend_break':    'purple',
        'volatility_change': 'teal',
    }

    for i, eid in enumerate(id_list):
        ax = axes[i][0]
        epd = raw_pd[raw_pd[CONFIG['id_col']] == eid].sort_values(CONFIG['time_col'])
        epl = pl.from_pandas(epd)
        # Compute rolling band
        epl = epl.with_columns([
            pl.col(vc).rolling_mean(window_size=W, center=True, min_periods=3).alias('_rm'),
            pl.col(vc).rolling_std(window_size=W, center=True, min_periods=3).alias('_rs'),
        ])
        ep2 = epl.to_pandas()
        ax.plot(ep2[CONFIG['time_col']], ep2[vc], lw=0.8, color='steelblue', alpha=0.7, zorder=1)
        ax.fill_between(ep2[CONFIG['time_col']],
                        ep2['_rm'] - 2 * ep2['_rs'],
                        ep2['_rm'] + 2 * ep2['_rs'],
                        alpha=0.15, color='steelblue', label='±2σ rolling band')
        ax.plot(ep2[CONFIG['time_col']], ep2['_rm'], lw=1.2, color='navy',
                alpha=0.6, linestyle='--', label='rolling mean')

        if not temp_df.empty:
            ef = temp_df[(temp_df['id'] == eid) & (temp_df['col'] == vc)]
            for method, grp in ef.groupby('method', dropna=False):
                colour = method_colours.get(str(method), 'grey')
                ax.axvline(grp['time'].iloc[0], color=colour, alpha=0.5,
                           linestyle=':', linewidth=1.2, label=str(method))
                ax.scatter(grp['time'], grp['value'], color=colour, s=50, zorder=4)

        ax.set_title(str(eid), fontsize=10)
        ax.legend(fontsize=7, loc='upper left', ncol=3)
        ax.tick_params(axis='x', rotation=20, labelsize=8)

    plt.tight_layout()
    plt.show()

## Section 6 — Cross-Entity Anomalies

In [ ]:
# Per-entity aggregate statistics
agg_exprs = [pl.len().alias('n_rows')]
for vc in CONFIG['value_cols']:
    agg_exprs += [
        pl.col(vc).mean().alias(f'{vc}_mean'),
        pl.col(vc).std().alias(f'{vc}_std'),
        pl.col(vc).median().alias(f'{vc}_median'),
        (pl.col(vc).quantile(0.75) - pl.col(vc).quantile(0.25)).alias(f'{vc}_iqr'),
        pl.col(vc).skew().alias(f'{vc}_skew'),
        pl.col(vc).kurtosis().alias(f'{vc}_kurt'),
        (pl.col(vc).null_count() / pl.len() * 100).round(2).alias(f'{vc}_null_pct'),
    ]

entity_stats = df.group_by(CONFIG['id_col']).agg(agg_exprs)
print('Entity-level statistics:')
display(entity_stats)

In [ ]:
# Cross-entity Z-score: flag entities whose aggregate metrics are outliers vs peers
metric_cols = [c for c in entity_stats.columns if c != CONFIG['id_col']]
entity_stats_z = entity_stats.clone()

for col in metric_cols:
    col_data = entity_stats_z[col].cast(pl.Float64)
    mean_v = col_data.mean()
    std_v  = col_data.std()
    if std_v and std_v > 1e-10:
        entity_stats_z = entity_stats_z.with_columns(
            ((pl.col(col).cast(pl.Float64) - mean_v) / std_v).alias(f'{col}_z')
        )
    else:
        entity_stats_z = entity_stats_z.with_columns(
            pl.lit(0.0).alias(f'{col}_z')
        )

z_cols = [c for c in entity_stats_z.columns if c.endswith('_z')]
THRESH = CONFIG['cross_entity_zscore']

for row in entity_stats_z.to_dicts():
    for zc in z_cols:
        zval = row.get(zc, 0) or 0
        if abs(zval) > THRESH:
            base_metric = zc[:-2]  # strip _z
            # Classify by which metric is anomalous
            if 'null_pct' in zc:
                sev, atype = 8, 'possible_bug (high null rate)'
            elif 'std' in zc:
                sev, atype = 5, 'genuine_anomaly (high volatility)'
            elif 'mean' in zc and abs(zval) > 4:
                sev, atype = 7, 'possible_bug (systematic bias)'
            elif 'kurt' in zc:
                sev, atype = 6, 'genuine_anomaly (heavy tails)'
            else:
                sev, atype = 5, 'genuine_anomaly'

            register('cross_entity', {
                'id':    row[CONFIG['id_col']],
                'time':  '—',
                'col':   base_metric,
                'value': round(row.get(base_metric, 0) or 0, 4),
                'method': 'cross_entity_zscore',
                'score':  round(abs(zval), 3),
                'severity': sev,
                'anomaly_type': atype,
                'label': f'Cross-entity: {base_metric} Z={zval:.2f} (threshold ±{THRESH})',
            })

print(f'Cross-entity flags: {len(anomaly_registry["cross_entity"])}')

In [ ]:
# Cross-entity heatmap: Z-scores per entity per metric
if CONFIG['plot_anomalies'] and len(entity_stats_z) > 1:
    z_df_pd = entity_stats_z.select([CONFIG['id_col']] + z_cols).to_pandas()
    z_df_pd = z_df_pd.set_index(CONFIG['id_col'])
    z_df_pd.columns = [c.replace('_z', '') for c in z_df_pd.columns]

    # Drop constant-zero columns
    z_df_pd = z_df_pd.loc[:, (z_df_pd != 0).any(axis=0)]

    fig, ax = plt.subplots(figsize=(max(8, len(z_df_pd.columns) * 1.2),
                                    max(4, len(z_df_pd) * 0.8)))
    sns.heatmap(
        z_df_pd.astype(float),
        annot=True, fmt='.2f', cmap='RdBu_r',
        center=0, vmin=-5, vmax=5,
        linewidths=0.5, ax=ax
    )
    ax.set_title('Cross-Entity Z-Scores (metric vs population)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Metric'); ax.set_ylabel('Entity')
    plt.tight_layout()
    plt.show()

## Section 7 — Severity Scoring & Report

In [ ]:
# Consolidate all anomalies into a single DataFrame
all_records = [
    rec for category in anomaly_registry.values() for rec in category
]

if not all_records:
    print('No anomalies detected across any method.')
else:
    all_df = pd.DataFrame(all_records)

    # Ensure anomaly_type column exists everywhere
    if 'anomaly_type' not in all_df.columns:
        all_df['anomaly_type'] = 'unknown'
    all_df['anomaly_type'] = all_df['anomaly_type'].fillna('unknown')

    # Deduplicate: per (id, time, col) keep highest severity
    all_df = (
        all_df.sort_values('severity', ascending=False)
              .drop_duplicates(subset=['id', 'time', 'col', 'method'], keep='first')
              .reset_index(drop=True)
    )

    print(f'Total anomaly records (after deduplication): {len(all_df)}')
    display(all_df[['category','id','time','col','value','method','score','severity','label']].head(10))

In [ ]:
# Composite severity scoring
# composite = severity + 2*(method_count-1) + 1*(high_traffic) - 1*(isolated)

# Count how many distinct methods flagged each (id, time, col)
method_counts = (
    all_df.groupby(['id', 'time', 'col'])['method']
          .nunique()
          .rename('method_count')
          .reset_index()
)
all_df = all_df.merge(method_counts, on=['id', 'time', 'col'], how='left')

# High-traffic entity: top 25% by row count
id_row_counts = raw_pd.groupby(CONFIG['id_col']).size()
high_traffic_ids = set(id_row_counts[id_row_counts >= id_row_counts.quantile(0.75)].index)
isolated_ids     = set(id_row_counts[id_row_counts <= id_row_counts.quantile(0.25)].index)

all_df['composite_score'] = (
    all_df['severity']
    + 2 * (all_df['method_count'].fillna(1) - 1)
    + all_df['id'].apply(lambda x: 1 if x in high_traffic_ids else 0)
    - all_df['id'].apply(lambda x: 1 if x in isolated_ids else 0)
).clip(1, 10).round(1)

# Priority label
def priority_label(score):
    if score >= 8: return '🔴 DATA_PIPELINE_BUG'
    if score >= 6: return '🟠 DATA_ENTRY_ERROR'
    if score >= 4: return '🟡 GENUINE_ANOMALY'
    return '🔵 WARNING'

all_df['priority'] = all_df['composite_score'].apply(priority_label)

all_df = all_df.sort_values('composite_score', ascending=False).reset_index(drop=True)

print('Composite scoring applied.')
display(all_df[['priority','id','time','col','value','method','composite_score','label']].head(15))

In [ ]:
# Summary statistics by category and priority
summary = (
    all_df.groupby('priority')
          .agg(
              count=('id', 'count'),
              affected_entities=('id', 'nunique'),
              avg_composite_score=('composite_score', 'mean'),
          )
          .sort_values('avg_composite_score', ascending=False)
          .round(2)
)
print('=== SUMMARY ===')
display(summary)

In [ ]:
# Final report visualisation — 4-panel overview
if CONFIG['plot_anomalies'] and not all_df.empty:
    fig = plt.figure(figsize=(16, 11))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)
    fig.suptitle('Anomaly Detection — Final Report Overview', fontsize=14, fontweight='bold')

    priority_palette = {
        '🔴 DATA_PIPELINE_BUG': '#d62728',
        '🟠 DATA_ENTRY_ERROR':  '#ff7f0e',
        '🟡 GENUINE_ANOMALY':   '#bcbd22',
        '🔵 WARNING':           '#1f77b4',
    }

    # Top-left: count by priority (horizontal bar)
    ax1 = fig.add_subplot(gs[0, 0])
    cnt = all_df['priority'].value_counts().sort_index()
    colours = [priority_palette.get(p, 'grey') for p in cnt.index]
    ax1.barh(cnt.index, cnt.values, color=colours)
    ax1.set_title('Anomaly Count by Priority'); ax1.set_xlabel('Count')
    for i, v in enumerate(cnt.values):
        ax1.text(v + 0.1, i, str(v), va='center', fontsize=9)

    # Top-right: timeline scatter (anomalies over time)
    ax2 = fig.add_subplot(gs[0, 1])
    time_df = all_df[all_df['time'] != '—'].copy()
    time_df['_t'] = pd.to_datetime(time_df['time'], errors='coerce')
    time_df = time_df.dropna(subset=['_t'])
    if not time_df.empty:
        for prio, grp in time_df.groupby('priority'):
            ax2.scatter(grp['_t'], grp['id'], alpha=0.6,
                        s=grp['composite_score'] * 8,
                        color=priority_palette.get(prio, 'grey'), label=prio, zorder=3)
        ax2.set_title('Anomaly Timeline'); ax2.set_xlabel('Time'); ax2.set_ylabel('Entity')
        ax2.tick_params(axis='x', rotation=20, labelsize=8)
        ax2.legend(fontsize=7, loc='upper left')

    # Bottom-left: anomaly density heatmap (entity × day)
    ax3 = fig.add_subplot(gs[1, 0])
    if not time_df.empty:
        time_df['_day'] = time_df['_t'].dt.date
        heat = time_df.groupby(['id', '_day']).size().unstack(fill_value=0)
        if not heat.empty:
            sns.heatmap(heat, ax=ax3, cmap='YlOrRd', linewidths=0.3,
                        annot=len(heat.columns) <= 14, fmt='d', cbar_kws={'label': 'Count'})
            ax3.set_title('Anomaly Density (Entity × Day)')
            ax3.tick_params(axis='x', rotation=30, labelsize=8)

    # Bottom-right: pie chart by method
    ax4 = fig.add_subplot(gs[1, 1])
    method_counts_pie = all_df['method'].value_counts()
    ax4.pie(method_counts_pie.values, labels=method_counts_pie.index,
            autopct='%1.0f%%', startangle=140,
            textprops={'fontsize': 9})
    ax4.set_title('Detection Method Distribution')

    plt.show()

In [ ]:
# Generate prioritised markdown report
def build_report(all_df: pd.DataFrame, top_n: int) -> str:
    lines = []
    lines.append('# Anomaly Detection Report')
    lines.append(f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    lines.append(f'Dataset: `{CONFIG["file_path"]}` | {len(raw_pd):,} rows | '
                 f'{raw_pd[CONFIG["id_col"]].nunique()} entities')
    lines.append('')

    lines.append('## Executive Summary')
    for prio, grp in all_df.groupby('priority'):
        lines.append(f'- **{prio}**: {len(grp)} anomalies across {grp["id"].nunique()} entities')
    lines.append('')

    lines.append(f'## Top {min(top_n, len(all_df))} Prioritised Findings')
    lines.append('')

    top = all_df.head(top_n)
    for i, (_, row) in enumerate(top.iterrows(), 1):
        lines.append(f'### {i}. {row["priority"]} — Entity: `{row["id"]}`')
        lines.append(f'- **Time**: {row["time"]}')
        lines.append(f'- **Column**: `{row["col"]}` | **Value**: {row["value"]}')
        lines.append(f'- **Detection method**: `{row["method"]}`')
        lines.append(f'- **Score**: {row["score"]} | **Composite severity**: {row["composite_score"]}/10')
        lines.append(f'- **Details**: {row["label"]}')
        if 'anomaly_type' in row and pd.notna(row['anomaly_type']):
            lines.append(f'- **Classification**: {row["anomaly_type"]}')
        lines.append('')
        lines.append('---')
        lines.append('')

    return '\n'.join(lines)

if not all_df.empty:
    report_md = build_report(all_df, CONFIG['report_top_n'])
    print(report_md[:3000])
    print('...' if len(report_md) > 3000 else '')

    if CONFIG['output_report_path']:
        pathlib.Path(CONFIG['output_report_path']).write_text(report_md, encoding='utf-8')
        print(f'\nReport written to: {CONFIG["output_report_path"]}')
else:
    print('No anomalies to report.')